## StatefulSets mit PostgreSQL

Ein StatefulSet in Kubernetes ist dafür gedacht, Anwendungen zu betreiben, die sich ihren Zustand merken müssen – zum Beispiel Datenbanken oder Systeme mit persistentem Speicher.

Im Gegensatz zu einem normalen Deployment behandelt ein StatefulSet seine Pods nicht als austauschbare Einheiten. Stattdessen bekommt jeder Pod eine feste Identität, die über seine gesamte Lebensdauer erhalten bleibt.

Das bedeutet konkret:

* Jeder Pod hat einen stabilen Namen (z. B. `app-0`, `app-1`, `app-2`)
* Diese Identität bleibt auch bestehen, wenn der Pod neu gestartet oder auf einen anderen Node verschoben wird
* Pods werden in einer definierten Reihenfolge gestartet, gestoppt und skaliert
* Jeder Pod kann seinen eigenen persistenten Speicher (Volume) haben

Das ist besonders wichtig für Anwendungen wie Datenbanken, bei denen ein Pod immer wieder auf “seine” Daten zugreifen muss.

---

Beispiel basierend auf [StatefulSet](https://kubernetes.io/docs/concepts/workloads/controllers/statefulset/).

Einfache Postgresql Installation, wo StatefulSet erzeugt

In [ ]:
%%bash
helm repo add bitnami https://charts.bitnami.com/bitnami
helm repo update

helm upgrade --install backstage-postgresql bitnami/postgresql \
  --namespace db \
  --create-namespace \
  --set architecture=replication \
  --set auth.username=backstage \
  --set auth.password=backstage123 \
  --set auth.database=backstagedb \
  --set auth.replicationUsername=repl_user \
  --set auth.replicationPassword=repl_password123 \
  --set readReplicas.replicaCount=2

PostgreSQL läuft hier als **Primary/Replica-Setup**: `backstage-postgresql-primary-0` ist die schreibende Hauptdatenbank, und `backstage-postgresql-read-0` sowie `read-1` sind Read Replicas, die Daten vom Primary replizieren.

Schreibzugriffe gehen auf den Primary, während die Read Replicas für Lesezugriffe verwendet werden können; dadurch verteilst du Leselast, hast aber noch kein vollständiges automatisches HA-Failover wie bei einem spezialisierten PostgreSQL-HA-Setup.


In [ ]:
%%bash
kubectl get statefulsets,pods,services -n db

Mittels folgenden Befehlen kann auf die Datenbank, in einem separaten Shell Tab, zugegriffen werden

In [ ]:
%%bash
export POSTGRES_ADMIN_PASSWORD=$(kubectl get secret --namespace db backstage-postgresql -o jsonpath="{.data.postgres-password}" | base64 -d)
export POSTGRES_PASSWORD=$(kubectl get secret --namespace db backstage-postgresql -o jsonpath="{.data.password}" | base64 -d)

echo kubectl run backstage-postgresql-client --rm --tty -i --restart='Never' --namespace db --image registry-1.docker.io/bitnami/postgresql:latest --env="PGPASSWORD=$POSTGRES_PASSWORD" \
      --command -- psql --host backstage-postgresql -U backstage -d backstagedb -p 5432

Zum Testen legen wir eine Tabelle an und geben dieser dann wieder aus

    CREATE TABLE personen (
        id SERIAL PRIMARY KEY,
        name VARCHAR(100) NOT NULL,
        email VARCHAR(150),
        alter INTEGER
    );
    
    INSERT INTO personen (name, email, alter)
    VALUES
        ('Anna Meier', 'anna.meier@example.com', 28),
        ('Luca Keller', 'luca.keller@example.com', 34),
        ('Sara Huber', 'sara.huber@example.com', 22);



Bei der ReadOnly Version können wir nur Abfragen durchführen

    
    SELECT * FROM personen;

Hier die Zugriffsinformationen

In [ ]:
%%bash
export POSTGRES_ADMIN_PASSWORD=$(kubectl get secret --namespace db backstage-postgresql -o jsonpath="{.data.postgres-password}" | base64 -d)
export POSTGRES_PASSWORD=$(kubectl get secret --namespace db backstage-postgresql -o jsonpath="{.data.password}" | base64 -d)

echo kubectl run backstage-postgresql-client --rm --tty -i --restart='Never' --namespace db --image registry-1.docker.io/bitnami/postgresql:latest --env="PGPASSWORD=$POSTGRES_PASSWORD" \
      --command -- psql --host backstage-postgresql-read -U backstage -d backstagedb -p 5432

---
### Aufräumen

In [ ]:
%%bash
helm uninstall backstage-postgresql -n db
kubectl delete ns db